# Interpreting a fitted model

Which features it uses, whether depth is helping, and what the anchor is
actually worth on your data.

In [1]:
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split

from lairnet import LAIRNetRegressor
from lairnet import inspection as ins

X, y = make_friedman1(n_samples=600, noise=1.0, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)
model = LAIRNetRegressor(n_hidden=60, n_layers=12, random_state=0)
model.fit(X_train, y_train)

LAIRNetRegressor(n_hidden=60, n_layers=12, random_state=0)

## Feature importance

Applied to the fitted model on held-out data, so it measures what this model
uses rather than what the data contains.

In [2]:
importance = ins.permutation_importance(model, X_test, y_test,
                                        n_repeats=10, random_state=0)
for name, mean, std in importance.top(6):
    flag = "" if mean - std > 0 else "   (within noise)"
    print(f"{name:<12} {mean:+.4f} +/- {std:.4f}{flag}")

feature 3    +0.6931 +/- 0.0397
feature 0    +0.4900 +/- 0.0644
feature 1    +0.4049 +/- 0.0570
feature 4    +0.1672 +/- 0.0209
feature 2    +0.1291 +/- 0.0169
feature 7    +0.0035 +/- 0.0081   (within noise)


`x0`–`x4` are the informative ones, which is the right answer for this problem.

## Is depth helping?

Costs one forward pass, not `n_layers` refits.

In [3]:
curve = ins.depth_curve(model, X_test, y_test)
print("best depth :", curve.best_depth, "of", len(curve.depths))
print("one depth  :", f"{curve.scores[0]:.4f}")
print("all depths :", f"{curve.scores[-1]:.4f}")

best depth : 12 of 12
one depth  : 0.7300
all depths : 0.8369


## How much do the depths disagree?

In [4]:
agreement = ins.layer_agreement(model, X_test)
print("mean correlation between depths:", f"{agreement.mean_correlation:.3f}")

mean correlation between depths: 0.942


Near-uniform high correlation is expected rather than wrong: the alignment
impulse pulls every layer towards the same anchor, so it *makes* them
correlated. That is why the median aggregator is a robustness choice and not a
variance-reduction one.

## What is the anchor worth?

Refits at `align=0`, which removes the target-aware component entirely, and
scores both on the same held-out data.

In [5]:
ins.anchor_contribution(model, X_train, y_train, X_test, y_test)

{'with_anchor': 0.8368730952588113,
 'without_anchor': 0.7472182863881988,
 'gain': 0.0896548088706125}

Training and evaluation data are separate required arguments. An earlier
signature took one pair and scored both arms on it, which gave the ablation an
in-sample score and **reversed the conclusion**.

In [6]:
sweep = ins.align_sensitivity(model, X_train, y_train,
                              X_valid=X_test, y_valid=y_test)
for a, s in zip(sweep.align_values, sweep.scores):
    print(f"align={a:<5} {s:.4f}")
print("best:", sweep.best_align, " gain over no anchor:",
      f"{sweep.anchor_gain:+.4f}")

align=0.0   0.7472
align=0.1   0.7828
align=0.25  0.8415
align=0.5   0.8369
align=0.75  0.7511
align=0.9   0.7139
best: 0.25  gain over no anchor: +0.0943


## Everything at once

In [7]:
from lairnet.explain import explain_model

report = explain_model(model, X_train, y_train, X_test, y_test,
                       n_repeats=10, align_values=(0.0, 0.25, 0.5, 0.75),
                       random_state=0)
print(report.summary())

score on the evaluation data: 0.8369

features the model uses
  feature 3                +0.6931 +/- 0.0397 
  feature 0                +0.4900 +/- 0.0644 
  feature 1                +0.4049 +/- 0.0570 
  feature 4                +0.1672 +/- 0.0209 
  feature 2                +0.1291 +/- 0.0169 

depth
  best at depth 12 of 12
  aggregating all depths is worth +0.1069 over one
  depths correlate 0.942 on average, so aggregation recovers less variance than independent errors would give

anchor
  with anchor    0.8369
  without anchor 0.7472
  the anchor is worth +0.0897 here
  best alignment coefficient tested: 0.25 (fitted at 0.5)

numerics
  readout condition numbers 2.51e+05 to 5.68e+06
